# Stage 1.5 — Redact Diagnosis Information from the Latest Clinical Note

MIMIC discharge summaries often state the diagnosis outright (e.g. a `Discharge Diagnosis:`
section, or "consistent with pneumonia" in the narrative). Feeding that straight into Stage 2
(Information Extraction) and Stage 3 (Symptom Tree) would let the LLM read the answer instead of
reasoning from symptoms — this stage removes it first.

Only the **latest admission's** `clinical_note` is redacted. Prior-admission history is left
alone — it already exposes its diagnosis explicitly via structured fields
(`primary_icd_code` / `primary_dx_title` in `admission_history`), so redacting those note
excerpts too would not hide anything additional; that's intentional context, not a leak of the
current admission's answer.

**Three passes** (see `redact_diagnosis_agent` in `pipeline.py`):
1. Deterministic section stripping — removes `Discharge/Final/Primary/Admission Diagnosis`
   sections outright. No LLM, no risk of missing these.
2. Per-section LLM pass for inline narrative mentions (e.g. in Brief Hospital Course). Run
   **per section** rather than once on the whole note — verified empirically that a 7B model's
   recall on this task drops sharply with longer context (an obvious mention was caught 3/3 times
   in isolation but 0/3 times in the full note).
3. Deterministic keyword backstop using ONLY the admission's **primary** diagnosis title — used
   only to *remove* matching terms, never to inform any prediction. Added because even the
   per-section LLM pass wasn't exhaustive within a single section (missed a second, more subtly
   phrased mention next to one it caught). Deliberately scoped to `primary_dx_title` alone, not
   the full `ground_truth_dx_titles` list — using all coded diagnoses (often 20+ for a complex
   admission: comorbidities, complications, past conditions) was tried first and verified to gut
   legitimate Past Medical History / HPI content, not just the target diagnosis.

**This is a defense-in-depth measure, not a guarantee** — spot-check the sample diff below before
trusting it across the full cohort.

**Input:** `cohort.pkl` (Stage 1) | **Output:** one JSON file per patient in
`data/stage_01b_redact_notes/` + an index


## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

import pandas as pd
from pipeline import (
    LLMNotAvailableError,
    REDACTION_CHECKPOINT_JSON,
    check_llm,
    get_llm_config,
    load_cohort,
    load_redaction_checkpoint,
    print_pipeline_banner,
    redact_diagnosis_agent,
    save_redaction_checkpoint,
    save_redaction_results,
)

print_pipeline_banner()
LLM_CONFIG = get_llm_config()
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
print(f"LLM ready — {LLM_CONFIG.method_prefix()}: {model_info}")

cohort_df = load_cohort()
print(f"Patients to redact: {len(cohort_df)}")


## Redact Diagnosis Information

This runs one LLM call per note section (not one per note) for better recall on the small model —
expect several LLM calls per patient, so this stage takes noticeably longer than a single-call-per-
patient stage.

In [ ]:
checkpoint = load_redaction_checkpoint()
redacted = list(checkpoint.values())
done_patients = set(checkpoint.keys())
print(f"Resuming: {len(done_patients)} already done, {len(cohort_df) - len(done_patients)} remaining")

for _, row in cohort_df.iterrows():
    hadm_id = str(row["hadm_id"])
    patient_id = str(row["patient_id"])
    if patient_id in done_patients:
        continue

    print(f"Redacting patient={patient_id} hadm_id={hadm_id}...")
    try:
        result = redact_diagnosis_agent(
            clinical_note=row["clinical_note"],
            admission_id=hadm_id,
            patient_id=patient_id,
            config=LLM_CONFIG,
            primary_dx_title=row.get("primary_dx_title"),
        )
    except (TimeoutError, LLMNotAvailableError) as exc:
        save_redaction_checkpoint(redacted)
        raise type(exc)(
            f"{exc}\nCheckpoint saved ({len(redacted)} patients) → {REDACTION_CHECKPOINT_JSON}\n"
            "Re-run this cell to resume."
        ) from exc

    print(
        f"  sections={result['n_sections_redacted']} "
        f"llm_mentions={result['n_llm_mentions_redacted']} "
        f"keyword_backstop={result['n_keyword_backstop_hits']}"
    )
    redacted.append({
        "patient_id": patient_id,
        "admission_id": row["admission_id"],
        "subject_id": int(row["subject_id"]),
        "hadm_id": int(row["hadm_id"]),
        "primary_icd_code": row["primary_icd_code"],
        "primary_dx_title": row["primary_dx_title"],
        "diagnosis_redaction_method": result.get("_method", "unknown"),
        "redacted_note": result["redacted_note"],
        "redacted_sections": result["redacted_sections"],
        "llm_redacted_mentions": result["llm_redacted_mentions"],
        "keyword_backstop_hits": result["keyword_backstop_hits"],
        "n_sections_redacted": result["n_sections_redacted"],
        "n_llm_mentions_redacted": result["n_llm_mentions_redacted"],
        "n_keyword_backstop_hits": result["n_keyword_backstop_hits"],
        "original_char_count": result["original_char_count"],
        "redacted_char_count": result["redacted_char_count"],
    })
    done_patients.add(patient_id)
    save_redaction_checkpoint(redacted)

results_df = pd.DataFrame(redacted)
results_df[[
    "patient_id", "n_sections_redacted", "n_llm_mentions_redacted",
    "n_keyword_backstop_hits", "original_char_count", "redacted_char_count",
]]


## Spot-Check a Sample

Always eyeball a few before/after diffs — the redaction agent is a defense-in-depth measure over a
small model, not a guarantee. Look specifically for any diagnosis name that survived.

In [ ]:
sample = results_df.iloc[0]
original = cohort_df.loc[cohort_df["patient_id"] == sample["patient_id"], "clinical_note"].iloc[0]

print(f"Patient {sample['patient_id']} | ground truth: {sample['primary_dx_title']}")
print("=" * 60)
print("ORIGINAL:")
print(original[:2000])
print("\n" + "=" * 60)
print("REDACTED:")
print(sample["redacted_note"][:2000])

## Save Results

In [ ]:
out = save_redaction_results(results_df)
print(f"Saved → {out}")
print(
    "\nNext: stage_02_information_extraction.ipynb and stage_03_symptom_tree.ipynb now load "
    "these redacted notes instead of the raw cohort note — re-run them after this stage."
)